You are given user login events.

Find users who logged in for:

3 or more consecutive days

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

data = [
    (101, '2025-01-01'),
    (101, '2025-01-02'),
    (101, '2025-01-03'),
    (101, '2025-01-05'),

    (102, '2025-01-01'),
    (102, '2025-01-03'),
    (102, '2025-01-04'),

    (103, '2025-01-07'),
    (103, '2025-01-08'),
    (103, '2025-01-09'),
    (103, '2025-01-10')
]

df = spark.createDataFrame(data, ["user_id", "login_date"])

df = df.withColumn(
    "login_date",
    to_date(col("login_date"))
)

In [0]:
df.show()

In [0]:
windows_spec = Window.partitionBy("user_id")\
                     .orderBy("login_date")
df = df.withColumn("rn", row_number().over(windows_spec))


In [0]:
df = df.withColumn('grp', date_sub(col("login_date"), col("rn")))

In [0]:
result = df.groupBy("user_id","grp") \
    .agg(
        min("login_date").alias("start_date"),
        max("login_date").alias("end_date"),
        count("*").alias("streak_length")
    )\
        .filter(col("streak_length") >= 3)

In [0]:
result.show()